# Day 084 | Seismic Signal Classification (DL)

## 100 Days of ML for Geotechnical Engineering
**Phase:** Deep Learning | **Author:** Ripon Chandra Malo | **Date:** YYYY-MM-DD

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| 1D CNN | Convolution over time series | Sliding filter over seismic waveform |
| Synthetic signal generation | Parametric waveform creation | Earthquake / blast / noise models |
| Data augmentation | Noise + time shift during training | Improve robustness to sensor variation |
| Conv1d → MaxPool1d → Linear | CNN architecture for sequences | Feature extraction from waveform |
| Confusion matrix | Per-class classification accuracy | Misclassify blast as earthquake? |
| Activation visualization | Which waveform regions activate filters | Interpret what the CNN learned |

---

## Our Geotechnical Problem

Seismic monitoring systems around construction sites and mines need to automatically classify
incoming waveforms as **earthquake**, **blast**, or **background noise** in real time.
Manual review is slow — a 1D CNN can classify a 512-sample window in microseconds.

```
  Raw waveform (512 samples @ ~100 Hz)
  ────────────────────────────────────
  Earthquake  → damped sinusoidal burst (2–8 Hz)
  Blast       → sharp Gaussian spike + fast decay
  Noise       → random Gaussian (ambient vibration)

  1D CNN learns to extract discriminative features
  without any hand-crafted spectral analysis
```

## Part 1: Generate Synthetic Seismic Waveforms

In [ ]:
# ───────────────────────────────────────────────────────
# IMPORTS
# ───────────────────────────────────────────────────────

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

torch.manual_seed(42)
np.random.seed(42)

SIGNAL_LEN = 512    # samples per waveform
FS         = 100.0  # sampling frequency (Hz)
t          = np.linspace(0, SIGNAL_LEN / FS, SIGNAL_LEN)  # time axis

print(f"Signal length:     {SIGNAL_LEN} samples")
print(f"Sampling rate:     {FS} Hz")
print(f"Duration per clip: {SIGNAL_LEN / FS:.1f} s")
print(f"Classes:           Earthquake (0), Blast (1), Noise (2)")

In [ ]:
# ───────────────────────────────────────────────────────
# SIGNAL GENERATORS
# ───────────────────────────────────────────────────────

def generate_earthquake(n=100, length=512, fs=100.0):
    """Damped sinusoidal burst: 2–8 Hz, random onset, random decay."""
    t_arr = np.linspace(0, length / fs, length)
    signals = []
    for _ in range(n):
        freq    = np.random.uniform(2, 8)         # Hz
        onset   = np.random.uniform(0.5, 3.0)     # s
        decay   = np.random.uniform(0.5, 2.0)     # 1/s
        amp     = np.random.uniform(0.5, 1.5)
        signal  = amp * np.exp(-decay * np.maximum(t_arr - onset, 0)) \
                  * np.sin(2 * np.pi * freq * (t_arr - onset)) \
                  * (t_arr > onset)
        signal += np.random.normal(0, 0.05, length)  # light noise floor
        signals.append(signal)
    return np.array(signals)


def generate_blast(n=100, length=512, fs=100.0):
    """Sharp Gaussian spike followed by fast exponential decay."""
    t_arr = np.linspace(0, length / fs, length)
    signals = []
    for _ in range(n):
        peak    = np.random.uniform(0.5, 3.0)     # peak time (s)
        width   = np.random.uniform(0.02, 0.08)   # Gaussian width (s)
        decay   = np.random.uniform(3.0, 8.0)     # fast decay rate
        amp     = np.random.uniform(1.0, 2.0)
        # Primary spike
        signal  = amp * np.exp(-((t_arr - peak) ** 2) / (2 * width ** 2))
        # Coda tail
        signal += 0.3 * amp * np.exp(-decay * np.maximum(t_arr - peak, 0)) * (t_arr > peak)
        signal += np.random.normal(0, 0.05, length)
        signals.append(signal)
    return np.array(signals)


def generate_noise(n=100, length=512):
    """Random Gaussian — ambient background vibration."""
    signals = []
    for _ in range(n):
        amp    = np.random.uniform(0.05, 0.3)
        signal = np.random.normal(0, amp, length)
        signals.append(signal)
    return np.array(signals)


# Generate 300 total (100 per class)
eq_signals    = generate_earthquake(100, SIGNAL_LEN, FS)
blast_signals = generate_blast(100, SIGNAL_LEN, FS)
noise_signals = generate_noise(100, SIGNAL_LEN)

X_raw = np.vstack([eq_signals, blast_signals, noise_signals])  # (300, 512)
y_raw = np.array([0]*100 + [1]*100 + [2]*100)                   # 0=EQ, 1=Blast, 2=Noise

print(f"Dataset shape: {X_raw.shape}")
print(f"Classes: {np.unique(y_raw, return_counts=True)}")

In [ ]:
# ───────────────────────────────────────────────────────
# VISUALIZE ONE EXAMPLE PER CLASS
# ───────────────────────────────────────────────────────

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
examples  = [(eq_signals[0], 'Earthquake', 'steelblue'),
             (blast_signals[0], 'Blast', 'coral'),
             (noise_signals[0], 'Noise', 'green')]

for ax, (sig, label, color) in zip(axes, examples):
    ax.plot(t, sig, color=color, linewidth=0.8)
    ax.set_ylabel('Amplitude', fontsize=11)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-2.5, 2.5)

axes[-1].set_xlabel('Time (s)', fontsize=12)
plt.suptitle('Synthetic Seismic Waveforms (3 Classes)', fontsize=14)
plt.tight_layout()
plt.show()

---

## Part 2: Data Augmentation and DataLoader Preparation

In [ ]:
# ───────────────────────────────────────────────────────
# TRAIN/TEST SPLIT — preserve class balance
# ───────────────────────────────────────────────────────

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

print(f"Train: {X_train_np.shape}  Test: {X_test_np.shape}")
print(f"Train class counts: {np.unique(y_train_np, return_counts=True)[1]}")
print(f"Test  class counts: {np.unique(y_test_np,  return_counts=True)[1]}")

In [ ]:
# ───────────────────────────────────────────────────────
# DATA AUGMENTATION: noise injection + random time shift
# ───────────────────────────────────────────────────────

def augment_signals(X, noise_std=0.03, max_shift=10):
    """
    Apply two augmentations:
    1. Add Gaussian noise to simulate sensor variation
    2. Random circular time shift to handle timing offsets
    """
    X_aug = X.copy()
    for i in range(len(X_aug)):
        # Add noise
        X_aug[i] += np.random.normal(0, noise_std, X.shape[1])
        # Random shift
        shift = np.random.randint(-max_shift, max_shift + 1)
        X_aug[i] = np.roll(X_aug[i], shift)
    return X_aug


# Augment training set (double it)
X_aug = augment_signals(X_train_np)
X_train_final = np.vstack([X_train_np, X_aug])
y_train_final = np.concatenate([y_train_np, y_train_np])

print(f"After augmentation — Train: {X_train_final.shape}")

# Convert to tensors: shape (N, 1, L) for Conv1d
X_tr_t = torch.tensor(X_train_final[:, np.newaxis, :], dtype=torch.float32)
y_tr_t = torch.tensor(y_train_final, dtype=torch.long)
X_te_t = torch.tensor(X_test_np[:, np.newaxis, :],   dtype=torch.float32)
y_te_t = torch.tensor(y_test_np,   dtype=torch.long)

train_ds = TensorDataset(X_tr_t, y_tr_t)
test_ds  = TensorDataset(X_te_t, y_te_t)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

print(f"Input tensor shape: {X_tr_t.shape}  (N, channels=1, length=512)")

---

## Part 3: 1D CNN Architecture

In [ ]:
# ───────────────────────────────────────────────────────
# 1D CNN: Conv1d → ReLU → MaxPool1d × 2 → FC layers
# ───────────────────────────────────────────────────────

class SeismicCNN(nn.Module):
    """
    1D CNN for seismic waveform classification.

    Architecture:
      Conv1d(1, 32, 7) → ReLU → MaxPool1d(2)
      Conv1d(32, 64, 5) → ReLU → MaxPool1d(2)
      Flatten
      Linear(64 * 126, 128) → ReLU
      Linear(128, 3)   [3 classes]

    Input:  (batch, 1, 512)
    Output: (batch, 3)  raw logits
    """
    def __init__(self, n_classes=3, signal_len=512):
        super().__init__()

        self.conv1 = nn.Conv1d(in_channels=1,  out_channels=32, kernel_size=7, padding=3)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64, kernel_size=5, padding=2)
        self.pool2 = nn.MaxPool1d(kernel_size=2)

        # Compute flattened size
        # After pool1: length = signal_len // 2 = 256
        # After pool2: length = 256 // 2       = 128
        self._flat_size = 64 * (signal_len // 4)

        self.fc1 = nn.Linear(self._flat_size, 128)
        self.fc2 = nn.Linear(128, n_classes)

        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        # x: (batch, 1, 512)
        x = F.relu(self.conv1(x))   # (batch, 32, 512)
        x = self.pool1(x)            # (batch, 32, 256)
        x = F.relu(self.conv2(x))   # (batch, 64, 256)
        x = self.pool2(x)            # (batch, 64, 128)
        x = x.view(x.size(0), -1)   # (batch, 64*128 = 8192)
        x = F.relu(self.fc1(x))     # (batch, 128)
        x = self.dropout(x)
        x = self.fc2(x)             # (batch, 3)
        return x

    def extract_conv1_features(self, x):
        """Return first-layer feature maps for visualization."""
        return F.relu(self.conv1(x))


cnn = SeismicCNN(n_classes=3, signal_len=SIGNAL_LEN)
n_params = sum(p.numel() for p in cnn.parameters())
print(cnn)
print(f"\nTotal parameters: {n_params:,}")

---

## Part 4: Training with Validation

In [ ]:
# ───────────────────────────────────────────────────────
# TRAINING LOOP
# ───────────────────────────────────────────────────────

N_EPOCHS = 40
optimizer = torch.optim.Adam(cnn.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)

train_losses, val_losses = [], []
train_accs,   val_accs   = [], []

for epoch in range(N_EPOCHS):
    # --- Train ---
    cnn.train()
    ep_loss, correct, total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        logits = cnn(X_batch)
        loss   = criterion(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        ep_loss += loss.item() * X_batch.size(0)
        preds    = logits.argmax(dim=1)
        correct += (preds == y_batch).sum().item()
        total   += X_batch.size(0)
    train_losses.append(ep_loss / total)
    train_accs.append(correct / total)

    # --- Validate ---
    cnn.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            logits  = cnn(X_batch)
            v_loss += criterion(logits, y_batch).item() * X_batch.size(0)
            preds   = logits.argmax(dim=1)
            v_correct += (preds == y_batch).sum().item()
            v_total   += X_batch.size(0)
    val_losses.append(v_loss / v_total)
    val_accs.append(v_correct / v_total)
    scheduler.step()

    if epoch % 5 == 0 or epoch == N_EPOCHS - 1:
        print(f"Epoch {epoch:2d}/{N_EPOCHS} | "
              f"Train Loss={train_losses[-1]:.4f} Acc={train_accs[-1]:.3f} | "
              f"Val Loss={val_losses[-1]:.4f} Acc={val_accs[-1]:.3f}")

print(f"\nFinal test accuracy: {val_accs[-1]*100:.1f}%")

In [ ]:
# ───────────────────────────────────────────────────────
# PLOT LOSS AND ACCURACY CURVES
# ───────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, 'b-', linewidth=1.5, label='Train')
axes[0].plot(val_losses,   'r-', linewidth=1.5, label='Val')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('CrossEntropy Loss', fontsize=12)
axes[0].set_title('Loss Curves — Seismic CNN', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].plot([a * 100 for a in train_accs], 'b-', linewidth=1.5, label='Train')
axes[1].plot([a * 100 for a in val_accs],   'r-', linewidth=1.5, label='Val')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Accuracy Curves — Seismic CNN', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].set_ylim(0, 105)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Part 5: Confusion Matrix and Classification Report

In [ ]:
# ───────────────────────────────────────────────────────
# EVALUATE ON TEST SET — CONFUSION MATRIX
# ───────────────────────────────────────────────────────

CLASS_NAMES = ['Earthquake', 'Blast', 'Noise']

cnn.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        preds = cnn(X_batch).argmax(dim=1)
        all_preds.extend(preds.numpy())
        all_true.extend(y_batch.numpy())

cm = confusion_matrix(all_true, all_preds)

# Plot confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(3))
ax.set_yticks(range(3))
ax.set_xticklabels(CLASS_NAMES, fontsize=12)
ax.set_yticklabels(CLASS_NAMES, fontsize=12)
ax.set_xlabel('Predicted', fontsize=13)
ax.set_ylabel('True', fontsize=13)
ax.set_title('Confusion Matrix — Seismic Signal Classification', fontsize=13)

for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=15, color='white' if cm[i, j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(all_true, all_preds, target_names=CLASS_NAMES))

---

## Part 6: Visualize First-Layer Activations

In [ ]:
# ───────────────────────────────────────────────────────
# VISUALIZE WHAT CONV1 DETECTS IN EACH SIGNAL TYPE
# ───────────────────────────────────────────────────────

cnn.eval()

# Take one example of each class
examples_idx = [
    (np.where(y_test_np == 0)[0][0], 'Earthquake', 'steelblue'),
    (np.where(y_test_np == 1)[0][0], 'Blast',      'coral'),
    (np.where(y_test_np == 2)[0][0], 'Noise',      'green'),
]

fig, axes = plt.subplots(3, 2, figsize=(16, 10))

for row, (idx, label, color) in enumerate(examples_idx):
    signal = X_test_np[idx]                                         # (512,)
    x_t    = torch.tensor(signal[np.newaxis, np.newaxis, :], dtype=torch.float32)

    with torch.no_grad():
        feature_maps = cnn.extract_conv1_features(x_t)             # (1, 32, 512)

    # --- Left: raw signal ---
    axes[row, 0].plot(t, signal, color=color, linewidth=0.8)
    axes[row, 0].set_title(f'{label} — Raw Waveform', fontsize=12)
    axes[row, 0].set_ylabel('Amplitude', fontsize=10)
    axes[row, 0].grid(True, alpha=0.3)
    if row == 2:
        axes[row, 0].set_xlabel('Time (s)', fontsize=11)

    # --- Right: first 8 conv1 feature maps as heatmap ---
    fm = feature_maps[0, :8, :].numpy()                            # (8, 512)
    im = axes[row, 1].imshow(fm, aspect='auto', cmap='viridis',
                              extent=[0, SIGNAL_LEN / FS, 0, 8])
    axes[row, 1].set_title(f'{label} — Conv1 Activations (filters 0–7)', fontsize=12)
    axes[row, 1].set_ylabel('Filter index', fontsize=10)
    if row == 2:
        axes[row, 1].set_xlabel('Time (s)', fontsize=11)
    plt.colorbar(im, ax=axes[row, 1], fraction=0.03)

plt.suptitle('First Conv Layer Activations per Signal Class', fontsize=14)
plt.tight_layout()
plt.show()

print("Observation: Each class activates different filters at different times.")
print("Earthquakes: sustained activations; Blasts: sharp localized peak; Noise: diffuse.")

In [ ]:
# ───────────────────────────────────────────────────────
# AVERAGE ACTIVATION ENERGY PER CLASS
# ───────────────────────────────────────────────────────

cnn.eval()
avg_energy = {c: np.zeros(SIGNAL_LEN) for c in range(3)}
count      = {c: 0 for c in range(3)}

with torch.no_grad():
    for i, (sig, lbl) in enumerate(zip(X_test_np, y_test_np)):
        x_t = torch.tensor(sig[np.newaxis, np.newaxis, :], dtype=torch.float32)
        fm  = cnn.extract_conv1_features(x_t)   # (1, 32, 512)
        energy = fm[0].abs().mean(dim=0).numpy()  # mean over filters → (512,)
        avg_energy[lbl] += energy
        count[lbl] += 1

for c in range(3):
    avg_energy[c] /= count[c]

fig, ax = plt.subplots(figsize=(12, 4))
for c, (label, color) in enumerate(zip(CLASS_NAMES, ['steelblue', 'coral', 'green'])):
    ax.plot(t, avg_energy[c], color=color, linewidth=1.5, label=label)
ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Mean |Activation|', fontsize=12)
ax.set_title('Average Conv1 Activation Energy per Class', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## Practice Exercises

### Exercise 1: Deeper CNN
Add a third conv layer `Conv1d(64, 128, 3)` → `MaxPool1d(2)`. Does accuracy improve on 300 samples?

### Exercise 2: Real Spectrogram Features
Compute a short-time Fourier transform (STFT) of each signal and pass the 2D spectrogram through a 2D CNN.

### Exercise 3: Harder Augmentation
Add random amplitude scaling and polarity flip to the augmentation. How does it affect val accuracy?

In [ ]:
# EXERCISE SPACE

# Exercise 1: Deeper CNN
# class DeepSeismicCNN(nn.Module):
#     def __init__(self, n_classes=3, signal_len=512):
#         super().__init__()
#         self.conv1 = nn.Conv1d(1, 32, 7, padding=3)
#         self.pool1 = nn.MaxPool1d(2)
#         self.conv2 = nn.Conv1d(32, 64, 5, padding=2)
#         self.pool2 = nn.MaxPool1d(2)
#         self.conv3 = nn.Conv1d(64, 128, 3, padding=1)
#         self.pool3 = nn.MaxPool1d(2)
#         flat = 128 * (signal_len // 8)
#         self.fc1 = nn.Linear(flat, 128)
#         self.fc2 = nn.Linear(128, n_classes)

# Exercise 2: STFT spectrogram
# import scipy.signal
# f, t_spec, Zxx = scipy.signal.stft(signal, fs=100, nperseg=64)
# spectrogram = np.abs(Zxx)  # 2D, feed into Conv2d

# Exercise 3: Amplitude + polarity augmentation
# def augment_v2(X):
#     X_aug = X.copy()
#     for i in range(len(X_aug)):
#         scale    = np.random.uniform(0.5, 2.0)
#         polarity = np.random.choice([-1, 1])
#         X_aug[i] = polarity * scale * X_aug[i]
#     return X_aug

---

## Key Takeaways

1. **1D CNNs are natural fits for seismic waveforms.** They learn local temporal patterns (spikes, oscillations) without hand-crafted features like FFT.

2. **Signal generation must be physically consistent.** Earthquake waveforms (sinusoidal burst + decay) and blast waveforms (sharp spike) reflect real geophysical behavior.

3. **Augmentation is critical with small datasets.** Doubling the training set with noise injection and time shifts improves robustness to sensor variability.

4. **Conv1 activations are interpretable.** Earthquake filters respond to sustained oscillations; blast filters fire at sharp energy peaks; noise shows no coherent pattern.

5. **Confusion between classes reveals real-world difficulty.** Blast misclassification as earthquake (or vice versa) has major consequences — calibrate thresholds accordingly.

---

## How This Connects to ML

| CNN Concept | Role Here | Traditional Alternative |
|---|---|---|
| Conv1d kernel | Learned temporal filter | Hand-designed bandpass filter |
| MaxPool1d | Temporal downsampling | Decimation / strided window |
| ReLU | Non-linearity | No equivalent (linear methods can't capture bursts) |
| CrossEntropyLoss | Multi-class classification | SVM with RBF kernel |
| Dropout | Regularization | L2 on SVM margin |
| Feature map visualization | Activation-based interpretation | SHAP / LIME on extracted features |

**Tomorrow:** The Phase 4 capstone — predicting ground motion parameters (PGA, PGV, Sa) with LSTM and comparing to XGBoost.

---

*Day 084 of 100 — The ground speaks; now we can listen!*